In [12]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# [실습] Advanced RAG

RAG의 기본 베이스 체인에서 시작하여, 다양한 기능을 추가해 보겠습니다.   

### 라이브러리 설치  

랭체인 관련 라이브러리와 벡터 데이터베이스 라이브러리를 설치합니다.   

In [2]:
%pip install jsonlines openai langchain==0.3.27 langchain-openai langchain-community==0.3.27 langchain-chroma tiktoken rank_bm25 pymupdf kiwipiepy

In [3]:
%pip install jsonlines openai langchain langchain-openai langchain-community langchain-chroma tiktoken rank_bm25 pymupdf kiwipiepy

In [4]:
# 핵심 + OpenAI + 유틸리티
%pip install langchain>=1.0.0 langchain-core>=1.0.0 langchain-openai langchain-text-splitters tiktoken python-dotenv

# 벡터스토어 (FAISS)
%pip install faiss-cpu

# 레거시 기능 사용 시 추가
%pip install langchain-classic

  Using cached langchain_core-1.4.0-py3-none-any.whl.metadata (4.5 kB)
  Using cached langchain_text_splitters-1.1.2-py3-none-any.whl.metadata (3.3 kB)
Using cached langchain_core-1.4.0-py3-none-any.whl (548 kB)
Using cached langchain_text_splitters-1.1.2-py3-none-any.whl (35 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 0.3.86
    Uninstalling langchain-core-0.3.86:
      Successfully uninstalled langchain-core-0.3.86
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 0.3.11
    Uninstalling langchain-text-splitters-0.3.11:
      Successfully uninstalled langchain-text-splitters-0.3.11
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-openai 0.3.35 requires langchain-core<1.0.0,>=0.3.78, but you have langchain-core 1.4.0 which is incompatible.
langchain

### LLM과 임베딩 모델 불러오기

In [5]:
!pip install dotenv

In [6]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings

load_dotenv('.env', override=True)
if os.environ.get('OPENAI_API_KEY'):
    print('OpenAI API 키 확인')

llm = ChatOpenAI(model="gpt-4.1-mini", temperature = 0)
sllm = ChatOpenAI(model='gpt-4.1-nano', temperature = 0.7)
reasoning_llm = ChatOpenAI(model='gpt-5-mini')

embeddings = OpenAIEmbeddings(model = 'text-embedding-3-large', chunk_size=100)

OpenAI API 키 확인


reports.zip 파일을 압축 해제합니다.

In [5]:
import zipfile

with zipfile.ZipFile('reports.zip', 'r') as zip_ref:
    zip_ref.extractall('.')

## 데이터 불러오기

폴더에 포함된 문서들을 pdf 로더로 불러옵니다.   
glob 라이브러리를 사용합니다.

In [7]:
from langchain_core.documents import Document
from glob import glob

# 모든 PDF 파일을 glob으로 찾음
pdf_files = glob("reports/*.pdf")
pdf_files

['reports/사업보고서(2025.03.11).pdf',
 'reports/반기보고서(2024.08.14).pdf',
 'reports/분기보고서(2024.11.14).pdf']

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader

# 각 PDF 파일에서 페이지별로 내용을 불러와 하나로 합침
all_papers=[]

for i, path_paper in enumerate(pdf_files):
    loader = PyMuPDFLoader(path_paper)
    pages = loader.load()
    # 문서가 주어지면 Document List를 출력
    # Page별 저장 --> 청킹을 다시 수행

    doc = Document(page_content='', metadata = {'index':i, 'source':pages[0].metadata['source']})
    for page in pages:
        doc.page_content += page.page_content +' '

    doc.page_content = doc.page_content.replace('\n', ' ')
    for _ in range(10):
        doc.page_content = doc.page_content.replace('  ', ' ')
        doc.page_content = doc.page_content.replace('..', '.')

    all_papers.append(doc)

print(len(all_papers))
all_papers[0].page_content[:1000]

3


'목 차 사 업 보 고 서.1 【 대표이사 등의 확인 】.2 I. 회사의 개요.3 1. 회사의 개요.3 2. 회사의 연혁.5 3. 자본금 변동사항.8 4. 주식의 총수 등.9 5. 정관에 관한 사항.10 II. 사업의 내용.12 1. 사업의 개요.12 2. 주요 제품 및 서비스.13 3. 원재료 및 생산설비.17 4. 매출 및 수주상황.20 5. 위험관리 및 파생거래.21 6. 주요계약 및 연구개발활동.23 7. 기타 참고사항.25 III. 재무에 관한 사항.35 1. 요약재무정보.35 2. 연결재무제표.38 2-1. 연결 재무상태표.38 2-2. 연결 손익계산서.39 2-3. 연결 포괄손익계산서.40 2-4. 연결 자본변동표.40 2-5. 연결 현금흐름표.41 3. 연결재무제표 주석.43 1. 지배기업의 개요 (연결) .43 2. 연결재무제표 작성기준 및 중요한 회계정책 (연결).43 3. 중요한 판단과 추정 불확실성의 주요 원천 (연결) .71 4. 영업부문 (연결) .73 5. 현금및현금성자산 (연결).76 6. 사용이 제한되거나 담보로 제공된 금융자산 (연결).76 7. 범주별 금융상품 (연결) .77 8. 금융상품의 공정가치 (연결) .80 9. 매출채권 및 미수금 (연결) .85 10. 재고자산 (연결) .87 11. 공정가치측정금융자산 (연결).88 12. 관계기업투자주식 (연결).91 13. 종속기업 (연결) .94 14. 유형자산 (연결) .102 15. 무형자산 (연결) .104 16. 투자부동산 (연결).109 17. 리스 (연결) .111 18. 리스부채 (연결) .113 19. 금융리스채권 (연결).114 20. 충당부채 (연결) .115 21. 퇴직급여제도 (연결).119 22. 우발채무와 약정사항 (연결) .123 23. 납입자본 (연결) .126 24. 이익잉여금 (연결).127 25. 기타자본항목 (연결).128 26. 매출액 (연결).130 27. 비용의 성격별 분류 등 (연결).132 28. 판매비와 관리비 (연결).134 2

In [ ]:
all_papers

[Document(metadata={'index': 0, 'source': 'reports\\반기보고서(2024.08.14).pdf'}, page_content='목 차 반 기 보 고 서.1 【 대표이사 등의 확인 】.2 I. 회사의 개요.3 1. 회사의 개요.3 2. 회사의 연혁.5 3. 자본금 변동사항.8 4. 주식의 총수 등.9 5. 정관에 관한 사항.10 II. 사업의 내용.12 1. 사업의 개요.12 2. 주요 제품 및 서비스.13 3. 원재료 및 생산설비.17 4. 매출 및 수주상황.19 5. 위험관리 및 파생거래.20 6. 주요계약 및 연구개발활동.22 7. 기타 참고사항.25 III. 재무에 관한 사항.34 1. 요약재무정보.34 2. 연결재무제표.37 2-1. 연결 재무상태표.37 2-2. 연결 손익계산서.38 2-3. 연결 포괄손익계산서.39 2-4. 연결 자본변동표.39 2-5. 연결 현금흐름표.40 3. 연결재무제표 주석.42 1. 지배기업의 개요 (연결) .42 2. 연결재무제표 작성기준 및 중요한 회계정책 (연결).42 3. 중요한 판단과 추정 불확실성의 주요 원천 (연결) .43 4. 영업부문 (연결) .44 5. 범주별 금융상품 (연결) .46 6. 공정가치 (연결) .48 7. 공정가치측정금융자산 (연결).51 8. 관계기업투자주식 (연결).54 9. 종속기업 (연결) .58 10. 유형자산 (연결) .67 11. 무형자산 (연결) .68 12. 투자부동산 (연결).69 13. 리스 (연결) .71 14. 리스부채 (연결) .73 15. 금융리스채권 (연결).74 16. 충당부채 (연결) .76 17. 재무위험관리 (연결).79 18. 우발채무와 약정사항 (연결) .81 19. 납입자본 (연결) .83 20. 이익잉여금 (연결).84 21. 기타자본항목 (연결).85 22. 매출액 (연결).86 23. 판매비와 관리비 (연결).89 24. 기타수익 및 기타비용 (연결).91 25. 금융수익 및 금융비용 (연결).92 26. 법인세

글자 수와 토큰 수를 확인해 보겠습니다.

In [8]:
import tiktoken

encoder = tiktoken.encoding_for_model('gpt-4.1-mini') # 4.1, 5, 4o 모두 동일
for paper in all_papers:
    print(len(paper.page_content), len(encoder.encode(paper.page_content)), paper.metadata['source'])

510249 283113 reports/사업보고서(2025.03.11).pdf
345728 182859 reports/반기보고서(2024.08.14).pdf
212976 123712 reports/분기보고서(2024.11.14).pdf


# 토큰 단위로 청킹하기   

tiktoken이나 huggingface를 이용해 토큰 단위 청킹을 수행합니다.

In [9]:
!pip install langchain_text_splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken
token_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-4.1-mini",
    chunk_size=2000,
    chunk_overlap=400,
)
# from_huggingface_tokenizer : 허깅페이스 모델에서 토크나이저 가져오기

token_chunks = token_splitter.split_documents(all_papers)
print(len(token_chunks))

371


벡터 DB를 구성합니다.   

In [10]:
from langchain_chroma import Chroma
from tqdm import tqdm

Chroma().delete_collection()
db = Chroma(embedding_function=embeddings,
                           persist_directory="./chroma2",
                           collection_metadata={'hnsw:space':'l2'},
                           collection_name='finance',
                           )

db.add_documents(token_chunks)

retriever = db.as_retriever(search_kwargs={"k": 5})

# filter 옵션을 통해 특정 메타데이터를 가진 벡터만 검색 가능
# retriever = db.as_retriever(search_kwargs={"k": 5,"filter":{'author':'Hyungho Byun'}})

RAG 체인을 구현합니다.

In [11]:
retriever.invoke("SDS 2024년 매출액이 얼마야?")

[Document(id='07fab05e-ec3c-45db-bd7c-46771a248808', metadata={'source': 'reports/사업보고서(2025.03.11).pdf', 'index': 0}, page_content='675,110 7.8% Ⅰ. 납입자본 38,689 38,689 - 0.0% Ⅱ. 주식발행초과금 1,297,467 1,297,467 - 0.0% Ⅲ. 이익잉여금 7,995,006 7,446,854 548,152 7.4% Ⅳ. 기타자본항목 1,957 (125,001) 126,958 101.6% 비지배지분 372,330 317,562 54,768 17.2% 자본총계 9,705,449 8,975,571 729,878 8.1% 부채와 자본총계 13,238,330 12,321,025 917,305 7.4% 부채비율 36.4% 37.3% -0.9%p - 자기자본비율 73.3% 72.8% 0.5%p - 전자공시시스템 dart.fss.or.kr Page 293 나. 영업실적 [()는 부(-)의 수치임] ○ 2024년 매출은 전년대비 4.2% 증가한 13조 8,282억원, 영업이익은 12.7% 증가한 9,111억원을 기록하였고, 당기순이익은 12.6% 증가한 7,895억원을 기록하였습니다. 당사의 사업부문은 IT서비스와 물류로 나뉘며, 각 사업부문별 영업실적에 영향을 미친 요인과 내용 은 다음과 같습니다. 【사업부문별 영업실적 분석】 ○ IT서비스 사업부문 (단위 : 백만원, %) 구 분 제40기 (2024년) 제39기 (2023년) 증감 증감률 매출액 13,828,232 13,276,844 551,388 4.2% 매출원가 11,815,900 11,480,695 335,205 2.9% 매출총이익 2,012,332 1,796,149 216,183 12.0% 판매비와 관리비 1,101,235 987,918 113,317 11.5% 영업이익 911,097 808,231 102,866 12.7% 기타영업수익 56,645 74,712 (18,067) -

In [11]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate([
    ("user", '''당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 한국어로 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면, "정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}''')])
prompt.pretty_print()

================================ Human Message =================================

당신은 QA(Question-Answering)을 수행하는 Assistant입니다.
다음의 Context를 이용하여 Question에 한국어로 답변하세요.
정확한 답변을 제공하세요.
만약 모든 Context를 다 확인해도 정보가 없다면, "정보가 부족하여 답변할 수 없습니다."를 출력하세요.
---
Context: {context}
---
Question: {question}


In [12]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def format_docs(docs):
    return "\n---\n".join(["Content: " + doc.page_content for doc in docs])
    # join : 구분자를 기준으로 스트링 리스트를 하나의 스트링으로 연결

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

다양한 질문을 통해 결과를 확인해 보겠습니다.

In [13]:
private_questions = [
    '이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?',
    'IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?',
    '2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?',
    '2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?',
    'GPUaaS 서비스가 무엇입니까?',
    '회사의 연구개발 담당조직은 어떤 분야의 핵심 기술을 연구하고 있습니까?',
    '2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.'
]
result = rag_chain.batch(private_questions)
for i, ans in enumerate(result):
    print(f"Question: {private_questions[i]}")
    print(f"Answer: {ans}")
    print('---')

Question: 이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?
Answer: 이 회사의 사업은 크게 IT서비스 부문과 물류 부문으로 나눠집니다.
---
Question: IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?
Answer: IT서비스는 3개의 분야로 나눠집니다. 각각은 다음과 같습니다.

1. 클라우드 서비스  
2. SI (시스템 통합)  
3. ITO (IT 아웃소싱)  

이 세 분야를 통해 다양한 IT서비스를 제공하고 있습니다.
---
Question: 2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?
Answer: 2024년 IT서비스 사업부문의 매출 비중은 46.3%로, 2023년의 46.0%에서 0.3%p 증가하였습니다. 물류 사업부문의 매출 비중은 53.7%로, 2023년 54.0%에서 0.3%p 감소하였습니다. 

또한, IT서비스 내에서 클라우드 사업 매출 비중은 2023년 31%에서 2024년 36%로 확대되었습니다.

2025년에 대한 구체적인 매출 비중 수치는 제공되지 않았으나, IT서비스 부문은 클라우드와 AI 중심의 디지털 전환 서비스 확대를 통해 성장할 것으로 전망되며, 물류 부문은 신규 선박 투입에 따른 공급 증가와 미국 관세 정책 영향으로 물류 운임이 하락할 것으로 예상됩니다. 따라서 2025년에도 IT서비스 매출 비중이 증가하는 추세가 지속될 가능성이 높습니다.
---
Question: 2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?
Answer: 2024년 6월말 기준, 연결 재무제표의 유동자산은 8,436,908,715,388원입니다.
---
Question: GPUaaS 서비스가 무엇입니까?
Answer: GPUaaS 서비스는 생성형 AI를 위한 구독형 GPU 컴퓨팅 서비스로, 클라우드 환경에서 AI 모델 실험, 예측, 추론 등 빠른 연산 속도를 필요로 하는 업무를 위해 가상 GPU 서버를 제공하는 서비스입니다. 이 서비스는 AI 작업 처리 

# Multi-Query Retriever   
모호한 쿼리를 검색하는 대신, 다양한 관점에서 Paraphrazing한 쿼리를 사용할 수 있습니다.   
이 때, LLM의 도움을 받을 수 있습니다.

In [14]:
# Multi Query를 확인하기 위한 로깅
import logging

logging.basicConfig()
logging.getLogger('langchain.retrievers.multi_query').setLevel(logging.INFO)

In [15]:
from langchain_core.prompts import PromptTemplate

In [ ]:
# 핵심 + OpenAI + 유틸리티
%pip install langchain>=1.0.0 langchain-core>=1.0.0 langchain-openai langchain-text-splitters tiktoken python-dotenv

# 벡터스토어 (FAISS)
%pip install faiss-cpu

# 레거시 기능 사용 시 추가
%pip install langchain-classic

Note: you may need to restart the kernel to use updated packages.
   ---------------------------------------- 0.0/18.9 MB ? eta -:--:--
   -- ------------------------------------- 1.0/18.9 MB 8.3 MB/s eta 0:00:03
   ------ --------------------------------- 3.1/18.9 MB 9.6 MB/s eta 0:00:02
   ----------- ---------------------------- 5.2/18.9 MB 9.8 MB/s eta 0:00:02
   -------------- ------------------------- 6.8/18.9 MB 8.9 MB/s eta 0:00:02
   ------------------ --------------------- 8.9/18.9 MB 9.2 MB/s eta 0:00:02
   ----------------------- ---------------- 11.0/18.9 MB 9.3 MB/s eta 0:00:01
   --------------------------- ------------ 12.8/18.9 MB 9.4 MB/s eta 0:00:01
   ------------------------------- -------- 14.7/18.9 MB 9.2 MB/s eta 0:00:01
   ----------------------------------- ---- 16.8/18.9 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------  18.6/18.9 MB 9.3 MB/s eta 0:00:01
   ---------------------------------------- 18.9/18.9 MB 9.1 MB/s  0:00:02
Note: you may 

In [17]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
# from langchain.retrievers.multi_query import MultiQueryRetriever

rewrite_prompt = PromptTemplate(template = """
당신은 삼성SDS의 직원들을 대상으로 하는 챗봇입니다.
기업 공시 문서에 대한 질문이 주어집니다.
'우리 회사' 등의 표현은 회사명으로 변환하세요.

공시 문서는 '삼성SDS', '삼성에스디에스', 'Samsung SDS' 등의 표현이 혼재된 문서이므로
모든 표현을 하나씩 사용하세요.

해당 질문에 대한 정보를 검색하기 위해, 벡터 데이터베이스에 입력할 다양한 맥락의 질문을 생성하세요.
질문은 3개 생성하고, 한 줄에 질문 하나씩 출력하세요.
질문을 다각도로 분석하여, 다양한 검색 결과가 나오도록 구성해야 합니다.
---
원본 질문: {question}

""")

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=db.as_retriever(),
    llm=llm,
    prompt = rewrite_prompt,
)
# 질문이 주어지면, 질문을 3개로 분할
# 3*K 검색 (12)
# 합집합으로 최종 Context 만들기

In [18]:
len(multi_query_retriever.invoke("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요."))

4

In [18]:
multi_query_retriever.invoke("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.")

[Document(id='e3b06616-f0c1-4471-ad32-0bfa1f430ca5', metadata={'source': 'reports/분기보고서(2024.11.14).pdf', 'index': 2}, page_content='주당 현금배당금(원) 보통주 - 2,700 3,200 우선주 - - - 주당 주식배당(주) 보통주 - - - 우선주 - - - (단위: 회, %) 연속 배당횟수 평균 배당수익률 분기(중간)배당 결산배당 최근 3년간 최근 5년간 - 29 1.86 1.63 [지분증권의 발행 등과 관련된 사항] (기준일 : 2024년 09월 30일 ) (단위 : 원, 주) 주식발행 (감소)일자 발행(감소) 형태 발행(감소)한 주식의 내용 종류 수량 주당 주당발행 비고 전자공시시스템 dart.fss.or.kr Page 155 ※ 당사는 공시대상기간 중 자본금 변동사항이 없습니다. 나. 전환사채 등 발행현황 미상환 전환사채 발행현황 ※ 해당사항 없음 미상환 신주인수권부사채 등 발행현황 ※ 해당사항 없음 미상환 전환형 조건부자본증권 발행현황 ※ 해당사항 없음 다. 채무증권 발행실적 등 채무증권 발행실적 액면가액 (감소)가액 - - - - - - - (기준일 : 2024년 09월 30일 ) (단위 : 원, 주) 종류＼구분 회차 발행일 만기일 권면(전 자등록 )총액 전환대상 주식의 종류 전환청구 가능기간 전환조건 미상환사채 비고 전환비 율 (%) 전환가 액 권면(전 자등록 )총액 전환가능 주식수 - - - - - - - - - - - - 합 계 - - - - - - - - - - - (기준일 : 2024년 09월 30일 ) (단위 : 원, 주) 종류＼구분 회차 발행일 만기일 권면(전 자등록 )총액 행사대상 주식의 종류 신주인수 권 행사가 능기간 행사조건 미상환 사채 미행사 신주 인수권 비고 행사비 율 (%) 행사가 액 권면(전자 등록)총액 행사가능 주식수 - - - - - - - - - - - - 합 계 - - - - - - - - - - - (기준일 : 

In [19]:
rag_chain = (
    {"context": multi_query_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [20]:
questions = [
    '이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?',
    'IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?',
    '2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?',
    '2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?',
    'GPUaaS 서비스가 무엇입니까?',
    '회사의 연구개발 담당조직은 어떤 분야의 핵심 기술을 연구하고 있습니까?',
    '2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.'
]

result = rag_chain.batch(questions)
for i, ans in enumerate(result):
    print(f"Question: {questions[i]}")
    print(f"Answer: {ans}")
    print('---')

Question: 이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?
Answer: 이 회사의 사업은 크게 IT서비스 부문과 물류 부문, 두 개의 부문으로 나눠집니다.
---
Question: IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?
Answer: IT서비스는 3개의 분야로 나눠집니다. 각각은 다음과 같습니다.

1. 클라우드 서비스  
2. SI (시스템 통합)  
3. ITO (IT 아웃소싱)
---
Question: 2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?
Answer: 2024년과 2025년의 매출 비중 변화에 대한 정보는 다음과 같습니다.

- 2024년 삼성에스디에스의 전체 매출은 13조 8,282억원이며, 사업부문별 매출 비중은 IT서비스 부문이 46.3%, 물류 부문이 53.7%입니다.
- 2024년 IT서비스 부문 매출은 6조 4,014억원으로 전체 매출의 46.3%를 차지하였고, 물류 부문 매출은 7조 4,268억원으로 53.7%를 차지하였습니다.
- 2025년 국내 IT서비스 시장은 9.9% 성장할 것으로 전망되며, 글로벌 IT서비스 시장도 9.0% 성장할 것으로 예상되어 IT서비스 부문의 매출 비중이 증가할 가능성이 있습니다.
- 다만, 2025년 물류 부문은 신규 선박 투입에 따른 공급 증가와 미국 관세 정책에 따른 물동량 둔화로 물류 운임이 하락할 것으로 전망되어 물류 부문의 매출 비중은 상대적으로 감소할 수 있습니다.

요약하면, 2024년에는 물류 부문 매출 비중이 IT서비스 부문보다 다소 높았으나, 2025년에는 IT서비스 부문의 성장세가 더 커지면서 매출 비중이 IT서비스 부문 쪽으로 확대될 가능성이 있습니다. 정확한 2025년 매출 비중 수치는 제공된 자료에 명시되어 있지 않습니다.
---
Question: 2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?
Answer: 2024년 6월말 기준, 연결 재무제표의 유동자산은 8,436,908,715,3


### Ensemble Retriever

Ensemble Retriever는 서로 다른 리트리버를 결합하여 순위를 합산합니다.   
주로 Keyword Indexing 기반 검색인 BM25 검색과 Semantic 검색을 합친 Hybrid Search를 사용합니다.

In [20]:
from kiwipiepy import Kiwi

kiwi = Kiwi()
def kiwi_tokenize(text):
    return [token.form for token in kiwi.tokenize(text)]

kiwi_tokenize("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.")

['2021-2022',
 '년',
 '사이',
 '에',
 '있',
 '었',
 '던',
 '합병',
 '사례',
 '를',
 '모두',
 '파악',
 '하',
 '어',
 ',',
 '표',
 '로',
 '나타내',
 '세요',
 '.']

In [21]:
# from langchain.retrievers import BM25Retriever, EnsembleRetriever
from langchain_classic.retrievers import BM25Retriever, EnsembleRetriever


bm25_retriever = BM25Retriever.from_documents(token_chunks, preprocess_func = kiwi_tokenize)
bm25_retriever.k = 5

retriever = db.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever], weights=[0.5, 0.5]
)

In [26]:
questions = [
    '이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?',
    'IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?',
    '2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?',
    '2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?',
    'GPUaaS 서비스가 무엇입니까?',
    '회사의 연구개발 담당조직은 어떤 분야의 핵심 기술을 연구하고 있습니까?',
    '2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.'
]

In [24]:
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

In [27]:
result = rag_chain.batch(questions)
for i, ans in enumerate(result):
    print(f"Question: {questions[i]}")
    print(f"Answer: {ans}")
    print('---')

Question: 이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?
Answer: 이 회사의 사업은 크게 두 부문으로 나뉘며, 각각은 다음과 같습니다.

1. IT서비스 부문  
- IT기술 역량을 활용하여 고객 맞춤형 다양한 IT서비스를 제공  
- 클라우드 서비스, 시스템 통합(SI), IT 아웃소싱(ITO)으로 구성  
- 클라우드 서비스는 삼성 클라우드 플랫폼(SCP)을 기반으로 하며, 생성형 AI 서비스도 포함  
- SI는 ERP, SCM, CRM 등 업무시스템과 팩토리솔루션, 블록체인, 데이터분석플랫폼 등 솔루션 제공  
- ITO는 다양한 업종에 안정적이고 효율적인 IT 아웃소싱 서비스 제공

2. 물류 부문  
- 자체 개발한 물류 통합관리 플랫폼 Cello와 디지털 물류 플랫폼 Cello Square를 통해 종합 물류 서비스 제공  
- 국제운송, 내륙운송, 물류센터 운영, 이커머스, 프로젝트 물류 등 전 영역 물류 서비스  
- 고객 맞춤형 물류 컨설팅과 IT 기반 플랫폼을 통한 차별화된 물류 서비스 제공  
- 전 세계 36개국, 300여개 사이트, 6,000여명의 물류 전문가, 5,300여개 물류 파트너의 글로벌 네트워크 보유

따라서, 이 회사의 주요 사업부문은 IT서비스 부문과 물류 부문입니다.
---
Question: IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?
Answer: IT서비스는 3개의 분야로 나눠집니다. 각각은 다음과 같습니다.

1. 클라우드 서비스 (Cloud Service)  
2. SI (System Integration)  
3. ITO (IT Outsourcing)
---
Question: 2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?
Answer: 2024년과 2025년 매출 비중 변화에 대한 정보는 다음과 같습니다.

- 2024년 IT서비스 사업부문 매출 비중은 46.3%였고, 물류 사업부문 매출 비중은 53.7%였습니다.
- 2025년 전체 매출 

# Reranker
넓은 Retriever 검색 범위를 이용한 뒤, Reranker를 통해 개수를 줄일 수 있습니다.

In [26]:
%pip show torch

Name: torch
Version: 2.11.0+cu128
Summary: Tensors and Dynamic neural networks in Python with strong GPU acceleration
Home-page: https://pytorch.org
Author: 
Author-email: PyTorch Team <packages@pytorch.org>
License: BSD-3-Clause
Location: /usr/local/lib/python3.12/dist-packages
Requires: cuda-bindings, cuda-toolkit, filelock, fsspec, jinja2, networkx, nvidia-cudnn-cu12, nvidia-cusparselt-cu12, nvidia-nccl-cu12, nvidia-nvshmem-cu12, setuptools, sympy, triton, typing-extensions
Required-by: accelerate, fastai, peft, sentence-transformers, timm, torchdata, torchvision


In [27]:
%pip install pytorch

  Preparing metadata (setup.py) ... done
  error: subprocess-exited-with-error
  
  × python setup.py bdist_wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for pytorch
  Running setup.py clean for pytorch
Failed to build pytorch
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (pytorch)


In [28]:
%pip install langchain-community

  Using cached langchain_core-0.3.86-py3-none-any.whl.metadata (3.2 kB)
  Using cached langchain_text_splitters-0.3.11-py3-none-any.whl.metadata (1.8 kB)
Using cached langchain_core-0.3.86-py3-none-any.whl (461 kB)
Using cached langchain_text_splitters-0.3.11-py3-none-any.whl (33 kB)
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.4.0
    Uninstalling langchain-core-1.4.0:
      Successfully uninstalled langchain-core-1.4.0
  Attempting uninstall: langchain-text-splitters
    Found existing installation: langchain-text-splitters 1.1.2
    Uninstalling langchain-text-splitters-1.1.2:
      Successfully uninstalled langchain-text-splitters-1.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-classic 1.0.7 requires langchain-core<2.0.0,>=1.3.3, but you have langchain-core 0.3.86 which is incompatible.
langchain-c

In [30]:
# GPU가 있거나, RAM이 충분한 경우에만 아래 코드를 실행해 주세요.
from langchain.retrievers.document_compressors import CrossEncoderReranker
# from langchain_classic.retrievers.document_compressors import CrossEncoderReranker
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
# from langchain.retrievers import ContextualCompressionRetriever
from langchain_classic.retrievers import ContextualCompressionRetriever

import torch


# Top 20
rough_retriever = db.as_retriever(search_kwargs={"k": 20})

# 다국어 리랭커 모델 사용하기
model = HuggingFaceCrossEncoder(model_name="BAAI/bge-reranker-v2-m3")

compressor = CrossEncoderReranker(model=model, top_n=5)
# Cross Encoder : 질문과 Chunk를 같이 넣고 처리하는 Transformers 계열 모델
# 점수순으로 Top 5

compression_retriever = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=rough_retriever)
compressed_docs = compression_retriever.invoke("2021-2022년 사이에 있었던 합병 사례를 모두 파악하여, 표로 나타내세요.")

compressed_docs

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

[Document(id='95aeb0cf-9daa-4b8d-ac2e-81129d313aa5', metadata={'source': 'reports/사업보고서(2025.03.11).pdf', 'index': 0}, page_content='해외사업환산손익 0 0 0 19,634,917,199 19,634,917,199 (2,512,586,071) 17,122,331,128 전자공시시스템 dart.fss.or.kr Page 40 2-5. 연결 현금흐름표 연차배당 0 0 (185,640,446,400) 0 (185,640,446,400) (2,625,998,400) (188,266,444,800) 주식보상비용 0 0 0 0 0 0 0 주식선택권 행사 0 0 0 0 0 0 0 사업결합 0 0 0 0 0 0 0 종속기업에 대한 소유지분의 변동 등 0 0 0 0 0 (44,449,077) (44,449,077) 2022.12.31 (기말자본) 38,688,900,000 1,297,466,852,618 7,000,953,240,103 (113,915,687,676) 8,223,193,305,045 243,776,629,869 8,466,969,934,914 2023.01.01 (기초자본) 38,688,900,000 1,297,466,852,618 7,000,953,240,103 (113,915,687,676) 8,223,193,305,045 243,776,629,869 8,466,969,934,914 당기순이익 0 0 693,422,011,534 0 693,422,011,534 7,903,749,488 701,325,761,022 기타포괄손익-공정가치측정금융자산평가손익 0 0 0 33,710,472 33,710,472 (222,478,025) (188,767,553) 지분법자본변동 0 0 0 2,867,203,173 2,867,203,173 0 2,867,203,173 확정급여제도의 재측정요소 0 0 0 (56,109,678,025) (56,109,678,025) (3,1

# Contextual Retrieval    

Claude가 제안한 Contextual Retrieval은 전체 Context를 활용하여     
청크별 헤더를 추가하는 방법입니다.    

In [31]:
# 청크 초기화
token_chunks = token_splitter.split_documents(all_papers)
print(len(token_chunks))

371


In [32]:
context_prompt = ChatPromptTemplate(
    [
        ('user', '''
기업 공시 보고서의 전체 내용과, 그 중 일부 Chunk가 주어집니다.
주어진 Document의 일부인 Chunk에 대해
간결하고 관련성 있는 짧은 설명을 생성하세요.
청크만으로는 명확하지 않으나, 전체를 참고하여 파악할 수 있는 정보를 추가하여
청크의 내용이 더 명확해지도록 하는 2~4문장 길이의 Context를 생성하면 됩니다.
아래의 가이드라인을 참고하세요.

1. 텍스트 부분에서 논의된 주요 주제나 개념을 포함하세요.
2. 문서 전체의 문맥에서 관련 정보나 비교를 언급하세요.
3. 가능한 경우, 이 정보가 문서의 전체적인 주제나 목적과 어떻게 연관되는지를 설명하세요.
4. 중요한 정보를 제공하는 주요 항목과 수치를 포함하세요.
5. 답변은 한국어로 작성합니다.

답변은 간결하게 작성하세요.

# Input Format

- [Document]: `<document> {document} </document>`
- [Chunk]: `<chunk> {chunk} </chunk>`

Context:
        ''')
    ]
)

# context_chain = context_prompt | llm | StrOutputParser()
context_chain = context_prompt | reasoning_llm | StrOutputParser()

Context가 잘 생성됐는지 확인해 봅니다.

In [34]:
chunk = token_chunks[40]
doc = all_papers[chunk.metadata['index']].page_content[:10]
context = context_chain.invoke({'document':doc, 'chunk':chunk.page_content})
print(context)
print('========')
print(chunk.page_content)

이 청크는 금융자산·금융부채를 측정기준별(상각후원가, 기타포괄손익-공정가치, 당기손익-공정가치)로 분류한 금액을 공시한 표입니다. 당기 기준 금융자산 합계는 7,969,146,958천원(상각후원가 7,969,146,958 포함, 기타포괄손익 22,521,632천원, 당기손익 71,122,213천원)이며 전기 합계는 7,206,695,684천원(기타포괄손익 14,625,131천원, 당기손익 67,060,584천원)입니다. 금융부채는 당기 합계 2,291,657,590천원(상각후원가 1,308,065,147천원, 기타부채 983,592,443천원), 전기 합계 2,190,651,897천원으로 표시되며 ‘기타’에는 미수수익·대여금·보증금·장기금융상품 등이 포함되고 각주로 종업원 관련 비계약성 채무(22,586백만원 등)와 급여 관련 제외액(442,548백만원), 미지급배당금·예수보증금 등이 설명되어 있습니다.
공정가치측 정금융자산 상각후원가 측정금융자 산 기타포괄손 익-공정가 치측정 금 융자산 당기손익- 공정가치측 정금융자산 금융자산 1,668,766 ,407 　 　 1,668,766 ,407 4,355,281 ,663 　 　 4,355,281 ,663 　 22,521,63 2 22,922,21 3 45,443,84 5 1,720,313 ,463 　 　 1,720,313 ,463 15,076,14 0 　 　 15,076,14 0 209,709,2 85 　 48,200,00 0 257,909,2 85 8,062,790 ,803 금융자산, 공정가치 1,668,766 ,407 　 　 1,668,766 ,407 4,355,281 ,663 　 　 4,355,281 ,663 　 22,521,63 2 22,922,21 3 45,443,84 5 1,720,313 ,463 　 　 1,720,313 ,463 15,076,14 0 　 　 15,076,14 0 209,709,2 85 　 48,200,00 0 257,909,2 85 8,062,790 ,803 전기 (단위 : 천원) 　 금

이제 Context 추가 작업을 수행합니다.

In [37]:
from tqdm import tqdm

chunk_with_parents=[]

for i, chunk in enumerate(tqdm(token_chunks[:10])):
    doc = all_papers[chunk.metadata['index']].page_content[:1000]
    chunk_with_parents.append({'document':doc, 'chunk':chunk.page_content})
    # print('\n'+context)
    # print('---')

# 10개씩 실행 (API 여유시 배치 늘리기)
for i in tqdm(range(0, len(token_chunks[:10]), 2)):
    contexts = context_chain.batch(chunk_with_parents[i:min(i+2, len(token_chunks[:10]))])
    for j in range(i,min(i+2, len(token_chunks[:10]))):
        token_chunks[j].page_content = context + '\n\n' + token_chunks[j].page_content

100%|██████████| 5/5 [01:18<00:00, 15.75s/it]


수정된 청크를 이용해, 벡터 데이터베이스를 다시 구성합니다.

In [ ]:
print(len(token_chunks))

371


In [47]:
db = Chroma(embedding_function=embeddings,
                           persist_directory="./chroma_Web_ContextualRetrieval",
                           collection_metadata={'hnsw:space':'l2'},
                           collection_name='Report',
                           )

db.add_documents(token_chunks[:50])

['d226e992-8f8f-4c48-b041-518f358705f8',
 '84e4b6a2-2a37-44aa-aa2d-87b6b1c83c2b',
 'de212cae-af6b-4ffb-9d71-9c4cd21f1efe',
 '213aaf33-f823-4a1f-b53a-d252d0d8fc70',
 '7be89c90-a77d-479d-b9d3-07158a3ff952',
 'd0e03dd1-c0ef-45da-90b8-31a93f4ee9d3',
 '440d8e65-bd46-464c-98d4-dbeddcc2fffe',
 '7b374251-fe5d-42d5-93ee-b8dc25448660',
 'ac3bc112-66a9-426f-9a1e-fe302945fee9',
 '03ec6c4d-658a-4608-b5df-9c99dcbad59e',
 'adac5fc8-2eb9-4010-b0a6-4a7de708bdf6',
 'e91783cc-56a0-4e82-88e3-4ccfd4ad97e4',
 'd4e727b0-7fca-4599-896a-a89c18c30897',
 '5450c01d-e95e-41a0-965e-57e2bbf19c14',
 '8a39012f-515e-441e-b0ce-29629cffba53',
 'cad94449-cb6b-4c5d-9df7-220b21df849f',
 'adfe3131-5940-46e8-97e0-5f19d046e0fe',
 '79368976-7bf5-4bd4-8253-5b20d4683279',
 'e0fb1cf7-aa56-4b97-837a-5f2b20204b31',
 'e12b3e21-1174-4b4c-b912-5ef343b92f71',
 'ce96cdba-2028-4e10-b10b-a835f4d01cf7',
 '572a2d3c-6d2f-44f1-9208-49e53dcab914',
 '80e7d796-cf63-4fa1-bf66-d41bcd4604de',
 '4f4e729d-24a9-40e4-8753-ab61b1a3943c',
 'c2ea8848-9550-

Contextual Header를 이용하기 위해, BM25와 Semantic Search를 결합합니다.

In [48]:
bm25_retriever = BM25Retriever.from_documents(token_chunks, preprocess_func = kiwi_tokenize)
bm25_retriever.k = 5

retriever = db.as_retriever(search_kwargs={"k": 5})

ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, retriever], weights=[0.5, 0.5]
)

In [49]:
rag_chain = (
    {"context": ensemble_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

result = rag_chain.batch(questions)
for i, ans in enumerate(result):
    print(f"Question: {questions[i]}")
    print(f"Answer: {ans}")
    print('---')

Question: 이 회사의 사업은 크게 두 부문으로 나눠집니다. 각각은 어떤 부문입니까?
Answer: 이 회사의 사업은 크게 두 부문으로 나뉘며, 각각 IT서비스 부문과 물류 부문입니다.
---
Question: IT서비스는 몇 개의 분야로 나눠집니까? 각각은 무엇입니까?
Answer: IT서비스는 3개의 분야로 나눠집니다. 각각은 클라우드 서비스, SI(시스템 통합), ITO(IT 아웃소싱)입니다.
---
Question: 2024년과 2025년에 걸쳐, 매출 비중은 어떻게 변화했나요?
Answer: 2024년과 2025년의 매출 비중 변화에 관한 구체적인 수치는 제공된 Context에 명확히 나와 있지 않습니다. 다만, 2024년 매출 총액은 13조 8,282억원으로 전년 대비 4.2% 증가하였고, IT서비스와 물류 사업부문 매출 비중은 각각 약 46.3%와 53.7%로 나타났습니다. 또한, 국내외 IT서비스 및 물류 시장 규모가 2024년부터 2028년까지 연평균 성장률과 함께 증가할 것으로 전망되고 있으나, 2025년의 당사 매출 비중에 대한 구체적인 수치는 제공되지 않았습니다.

따라서 2024년과 2025년의 매출 비중 변화에 대해서는 정보가 부족하여 정확한 답변을 드리기 어렵습니다.
---
Question: 2024년 6월말 기준, 연결 재무제표의 유동자산은 얼마인가요?
Answer: 2024년 6월말 기준 연결 재무제표의 유동자산은 9,003,786,950,710원입니다.
---
Question: GPUaaS 서비스가 무엇입니까?
Answer: GPUaaS(GPU as a Service) 서비스는 클라우드 환경에서 AI 모델 실험, 예측, 추론 등 빠른 연산 속도를 필요로 하는 업무를 위해 제공되는 구독형 가상 GPU 서버 서비스입니다. 이 서비스는 AI 작업 처리 시간을 단축하고 데이터 처리량을 증가시키기 위해 고성능 멀티 노드 GPU 서버를 지원하며, 데이터 처리의 전 과정을 지원하는 다양한 AI 특화 상품을 보유하고 있습니다.
---
Q